# Build CloudWatch alarms and an EventBridge rule to catch silent Glue job failures

The on-call rotation got paged because Tuesday's overnight ETL succeeded at the AWS level but wrote 47 rows instead of the usual 50,000. Glue's built-in metrics did not catch it. You have one session to publish a custom row-count metric from the script, alarm on it, and prove the page would fire on the next quiet failure.

Your job is to extend the Glue ETL pattern from Lab 04 with a CloudWatch custom metric, an alarm on that metric, an EventBridge rule that catches the alarm state change, and an SNS topic that pages the on-call. Glue 4.0 PySpark, one good run and one broken run, no Glue crawler.

The four deliverables map to four checkpoints:
1. Glue ETL job runs SUCCEEDED on the good dataset and publishes at least one `labrun/OutputRowCount` datapoint.
2. A CloudWatch alarm exists on that metric with threshold 1000, period 60 seconds, 1 evaluation period, and `LessThanThreshold`.
3. An EventBridge rule on `aws.cloudwatch` `CloudWatch Alarm State Change` events for the alarm, routed to the SNS topic.
4. Re-running the job against a broken (50-row) dataset flips the alarm to `ALARM` and SNS publishes at least one message.

**Time.** About 50 minutes hands-on. Two Glue runs at the 10-minute billing minimum each plus alarm propagation dominate the wall clock.

**Cost.** This lab costs about 30 to 50 cents if everything goes on the first try. Two Glue ETL runs at 2 G.1X workers are the only line item that materially registers (about $0.15 each). CloudWatch alarms, custom metrics, EventBridge rules, and SNS publishes are all free at this volume; you would have to leave eleven alarms running to start paying for them. The cleanup cell at the bottom of this notebook deletes everything so the bill stops the moment you finish.

This lab maps to AWS DEA-C01 Domain 3: Data Operations and Support (22% of exam weight).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import csv
import getpass
import io
import json
import random
import time
from datetime import datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "cloudwatch-pipeline-monitoring"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[9].slug exactly
REGION = "us-east-1"  # all DEA-C01 labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource names. Bucket name appends the account ID for global uniqueness.
ROLE_NAME = f"labrun-{LAB_ID}-job-role"
INLINE_POLICY_NAME = f"labrun-{LAB_ID}-job-inline"
JOB_NAME = f"labrun-{LAB_ID}-job"
ALARM_NAME = f"labrun-{LAB_ID}-alarm"
RULE_NAME = f"labrun-{LAB_ID}-eb-rule"
TOPIC_NAME = f"labrun-{LAB_ID}-topic"
BUCKET_NAME = None  # resolved after STS once the account ID is known
TOPIC_ARN = None  # resolved after sns.create_topic
SUBSCRIPTION_ARN = None  # resolved after sns.subscribe
ALARM_ARN = None  # resolved after put_metric_alarm + describe_alarms

# Custom metric coordinates. The Glue script publishes one datapoint per run
# under Namespace=labrun, MetricName=OutputRowCount, Dimension JobName=JOB_NAME.
METRIC_NAMESPACE = "labrun"
METRIC_NAME = "OutputRowCount"
ALARM_THRESHOLD = 1000  # rows; below this we page
ALARM_PERIOD_SECONDS = 60
ALARM_EVAL_PERIODS = 1

# Seed config for the two datasets. random.seed makes the row counts predictable.
SEED = 42
GOOD_ROWS = 50000  # well above the 1000 threshold
BROKEN_ROWS = 50  # well below the 1000 threshold

# Default subscription endpoint: a Beeceptor-style HTTPS endpoint the student
# can swap. The lab does not require the subscription to confirm; SNS will
# return PendingConfirmation for HTTPS subs until the student visits the URL.
DEFAULT_SUBSCRIPTION_ENDPOINT = "https://labrun-cwpipe-test.free.beeceptor.com"


In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"All DEA-C01 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first IAM call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that we know the account ID. S3 bucket names
# must be globally unique.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest is module-level and in strict reverse-creation order. Lab 10 has
# no critical (hourly-billed) resources: CloudWatch alarms above the free
# 10-alarm allowance accrue dollars per month, not dollars per hour, and
# Glue ETL jobs bill only while running.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist (not just print a warning).
#
# labrun-checks 0.3.0 ships AWS adapters only for s3_bucket, iam_role,
# glue_database, glue_table, kinesis_stream. It does NOT yet ship adapters
# for glue_job, cloudwatch_alarm, eventbridge_rule, sns_topic, or
# sns_subscription. An inline _LabAdapter subclass extending
# AwsCleanupAdapter is defined in the cleanup cell at the bottom of this
# notebook and passed to run_cleanup. The atexit handler uses the wrapper
# if it has already been defined, otherwise falls back to the base adapter.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="sns_subscription",
        id="pending",  # rewritten after sns.subscribe returns a real ARN
        region=REGION,
        cli_delete_command="aws sns unsubscribe --subscription-arn <SUBSCRIPTION_ARN>",
    ),
    CleanupResource(
        type="sns_topic",
        id=TOPIC_NAME,
        region=REGION,
        cli_delete_command=f"aws sns delete-topic --topic-arn arn:aws:sns:{REGION}:<ACCOUNT_ID>:{TOPIC_NAME}",
    ),
    CleanupResource(
        type="eventbridge_rule",
        id=RULE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws events remove-targets --rule {RULE_NAME} --ids 1 && "
            f"aws events delete-rule --name {RULE_NAME}"
        ),
    ),
    CleanupResource(
        type="cloudwatch_alarm",
        id=ALARM_NAME,
        region=REGION,
        cli_delete_command=f"aws cloudwatch delete-alarms --alarm-names {ALARM_NAME}",
    ),
    CleanupResource(
        type="glue_job",
        id=JOB_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {JOB_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this is the
    safety net for kernel crashes. Errors are swallowed because atexit
    handlers must not raise.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter()) if "_LabAdapter" in globals() else run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before creating
    any new resources.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources or")
    print("collide on the role and job names. Run the cleanup cell at the")
    print("bottom of this notebook first, or remove them manually with the")
    print("AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Seed the good and broken datasets, build the IAM role, create the Glue ETL job, and run it on the good data

Four pieces of plumbing land in this task. The script lives in S3 (Glue requires `Command.ScriptLocation` to point at a real object at job-create time), the IAM role grants both the usual S3 actions and the less-obvious `cloudwatch:PutMetricData`, and the job reads its input prefix from `--INPUT_PREFIX` so the same job definition handles both the good run and the broken run in Task 4.

1. **Two datasets.** The seed cell builds a `good/` prefix with ~50,000 deterministic rows (well above the 1000 threshold) and a `broken/` prefix with ~50 rows (well below the threshold). Both have the same schema: `order_id`, `customer_id`, `amount_cents`, `currency`, `processor_name`.
2. **IAM role** named `labrun-cloudwatch-pipeline-monitoring-job-role` with the Glue trust policy. Attach the managed `AWSGlueServiceRole`. The inline policy grants `s3:GetObject`, `s3:PutObject`, `s3:ListBucket` on the lab bucket and its `/*` ARN, AND `cloudwatch:PutMetricData` on `*`. The PutMetricData grant is the one most students miss; without it the script silently fails to publish (CloudWatch swallows IAM denies on the metric path) and Checkpoint 1 fails on the missing datapoint instead of on a job failure.
3. **The PySpark transform script** uploaded to `s3://{bucket}/scripts/transform.py`. Defined as `PYSPARK_SCRIPT` in the cell below. The script reads CSV from the input prefix, writes Parquet to a sibling `out/` prefix, counts rows, and calls `boto3.client('cloudwatch').put_metric_data` to publish one datapoint under namespace `labrun`, metric `OutputRowCount`, with dimension `JobName=<job-name>`. The boto3 client inside the script uses the Glue execution role automatically (no explicit credentials), which is why the role needs `cloudwatch:PutMetricData`; this is the only `boto3.client()` in this lab that does not pass `aws_session_token` explicitly, because it does not need to.
4. **The Glue ETL job** itself, created with `glue.create_job`. `DefaultArguments` carries `--BUCKET_NAME` and `--INPUT_PREFIX=good`; the broken run in Task 4 overrides `--INPUT_PREFIX` to `broken` at `start_job_run` time without redefining the job.

After creating the role, sleep 15 seconds before creating the job. IAM role propagation is asynchronous and Glue validates the role at job-create time.


In [ ]:
# NBVAL_SKIP
# Task 1: Create the bucket, seed good/ + broken/, build the IAM role, upload
# the PySpark script, create the Glue job, and start the good-data run.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the S3 bucket with s3.create_bucket(Bucket=BUCKET_NAME).
# us-east-1 rejects LocationConstraint; other regions require it.

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# Deterministic synthetic CSVs.
random.seed(SEED)
COLUMNS = ["order_id", "customer_id", "amount_cents", "currency", "processor_name"]
CURRENCIES = ["USD", "USD", "USD", "EUR", "GBP"]


def _rows(count: int, prefix: str) -> bytes:
    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(COLUMNS)
    for i in range(count):
        writer.writerow([
            f"ord_{prefix}_{i:06d}",
            f"cust_{random.randint(1000, 9999)}",
            random.randint(100, 99999),
            random.choice(CURRENCIES),
            f"processor_{random.choice(['alpha', 'beta', 'gamma'])}",
        ])
    return buf.getvalue().encode("utf-8")


good_csv = _rows(GOOD_ROWS, "good")
broken_csv = _rows(BROKEN_ROWS, "broken")

# YOUR CODE: Upload the two CSV payloads:
#   s3.put_object(Bucket=BUCKET_NAME, Key="good/orders.csv", Body=good_csv)
#   s3.put_object(Bucket=BUCKET_NAME, Key="broken/orders.csv", Body=broken_csv)

print(f"Seeded s3://{BUCKET_NAME}/good/orders.csv ({GOOD_ROWS} rows)")
print(f"Seeded s3://{BUCKET_NAME}/broken/orders.csv ({BROKEN_ROWS} rows)")

# PySpark script. The boto3.client('cloudwatch') call inside the script uses
# the Glue execution role automatically; that is why the role needs
# cloudwatch:PutMetricData. This is the only boto3.client in this lab that
# does not pass aws_session_token, and it is intentional.
PYSPARK_SCRIPT = """
import sys
import boto3
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext

args = getResolvedOptions(sys.argv, ["JOB_NAME", "BUCKET_NAME", "INPUT_PREFIX"])
sc = SparkContext.getOrCreate()
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session
job = Job(glue_ctx)
job.init(args["JOB_NAME"], args)

bucket = args["BUCKET_NAME"]
prefix = args["INPUT_PREFIX"]
job_name = args["JOB_NAME"]
in_path = "s3://" + bucket + "/" + prefix + "/"
out_path = "s3://" + bucket + "/out/" + prefix + "/"

df = spark.read.option("header", "true").csv(in_path)
row_count = df.count()
df.write.mode("overwrite").parquet(out_path)

cw = boto3.client("cloudwatch", region_name="us-east-1")
cw.put_metric_data(
    Namespace="labrun",
    MetricData=[
        {
            "MetricName": "OutputRowCount",
            "Dimensions": [{"Name": "JobName", "Value": job_name}],
            "Value": float(row_count),
            "Unit": "Count",
        }
    ],
)
print("Published OutputRowCount=" + str(row_count) + " for JobName=" + job_name)

job.commit()
""".lstrip()

SCRIPT_KEY = "scripts/transform.py"
script_uri = f"s3://{BUCKET_NAME}/{SCRIPT_KEY}"

# YOUR CODE: Upload the script with
#   s3.put_object(Bucket=BUCKET_NAME, Key=SCRIPT_KEY, Body=PYSPARK_SCRIPT.encode("utf-8"))

print(f"Script uploaded: {script_uri}")

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "glue.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the IAM role with iam.create_role(
#   RoleName=ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(trust_policy),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).

iam.attach_role_policy(
    RoleName=ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole",
)

# Inline policy. Includes cloudwatch:PutMetricData on *; the metric path
# does not support resource-level grants. Without this, the script silently
# fails to publish and Checkpoint 1 fails on the missing datapoint.
bucket_arn = f"arn:aws:s3:::{BUCKET_NAME}"
inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:PutObject",
                "s3:ListBucket",
            ],
            "Resource": [bucket_arn, f"{bucket_arn}/*"],
        },
        {
            "Effect": "Allow",
            "Action": "cloudwatch:PutMetricData",
            "Resource": "*",
        },
    ],
}

# YOUR CODE: Attach the inline policy with iam.put_role_policy(
#   RoleName=ROLE_NAME,
#   PolicyName=INLINE_POLICY_NAME,
#   PolicyDocument=json.dumps(inline_policy),
# ).

print(f"Role created: {ROLE_NAME}")
print("Managed policy AWSGlueServiceRole attached.")
print("Inline policy attached (S3 + cloudwatch:PutMetricData).")

print("Your IAM role is stuck in traffic, give it 15 seconds...")
time.sleep(15)
print("Role is ready.")

role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"

# YOUR CODE: Create the Glue ETL job with glue.create_job(
#   Name=JOB_NAME,
#   Role=role_arn,
#   Command={"Name": "glueetl", "ScriptLocation": script_uri, "PythonVersion": "3"},
#   DefaultArguments={
#       "--BUCKET_NAME": BUCKET_NAME,
#       "--INPUT_PREFIX": "good",
#       "--enable-metrics": "true",
#   },
#   GlueVersion="4.0",
#   WorkerType="G.1X",
#   NumberOfWorkers=2,
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).

print(f"Glue ETL job created: {JOB_NAME}")

# YOUR CODE: Start the first run on the good dataset and capture the run id:
#   run = glue.start_job_run(JobName=JOB_NAME)  # uses default INPUT_PREFIX=good
#   run_id = run["JobRunId"]

print(f"Job run submitted on the good dataset; run id will be printed below.")
print("Asking Glue to spin up workers; this takes a minute or so...")

TERMINAL_STATES = {"SUCCEEDED", "FAILED", "STOPPED", "TIMEOUT"}
deadline = time.time() + 720  # 12 minutes
last_state = None
run_resp = None
while time.time() < deadline:
    run_resp = glue.get_job_run(JobName=JOB_NAME, RunId=run_id)
    state = run_resp["JobRun"]["JobRunState"]
    if state != last_state:
        print(f"  state: {state}")
        last_state = state
    if state in TERMINAL_STATES:
        break
    time.sleep(15)
else:
    raise RuntimeError(
        f"Job run {run_id} did not reach a terminal state within 12 minutes. "
        f"Inspect the run in the AWS Glue console."
    )

final_state = run_resp["JobRun"]["JobRunState"]
if final_state == "SUCCEEDED":
    exec_seconds = run_resp["JobRun"].get("ExecutionTime", 0)
    print(f"Good-data run finished SUCCEEDED in {exec_seconds} seconds.")
    print("Sleeping 60 seconds so the metric datapoint has time to land in CloudWatch...")
    time.sleep(60)
else:
    err = run_resp["JobRun"].get("ErrorMessage", "(no error message)")
    print(f"Run finished {final_state}: {err}")
    print("Fix the script or role, then re-run this cell.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: most-recent Glue job run is SUCCEEDED and CloudWatch has at
# least one labrun/OutputRowCount datapoint in the last 30 minutes with the
# JobName dimension equal to JOB_NAME.


def checkpoint_1(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        cw_client = boto3.client(
            "cloudwatch",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            runs = glue_client.get_job_runs(JobName=JOB_NAME, MaxResults=1)
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Glue job {JOB_NAME} does not exist. Run the Task 1 cell.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        job_runs = runs.get("JobRuns", [])
        if not job_runs:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Job {JOB_NAME} has no runs yet. The Task 1 cell starts a run; "
                    f"wait for it to finish and re-run this checkpoint."
                ),
            )
        latest = job_runs[0]
        state = latest.get("JobRunState", "UNKNOWN")
        if state != "SUCCEEDED":
            err = latest.get("ErrorMessage", "(no error message)")
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Most-recent run finished {state}: {err}. Inspect the run in "
                    f"the Glue console or CloudWatch Logs and rerun after fixing."
                ),
            )

        end_time = datetime.now(timezone.utc)
        start_time = end_time - timedelta(minutes=30)
        stats = cw_client.get_metric_statistics(
            Namespace=METRIC_NAMESPACE,
            MetricName=METRIC_NAME,
            Dimensions=[{"Name": "JobName", "Value": JOB_NAME}],
            StartTime=start_time,
            EndTime=end_time,
            Period=60,
            Statistics=["Sum", "Maximum"],
        )
        datapoints = stats.get("Datapoints", [])
        if not datapoints:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No CloudWatch datapoints found for Namespace={METRIC_NAMESPACE!r}, "
                    f"MetricName={METRIC_NAME!r}, JobName={JOB_NAME!r} in the last 30 "
                    f"minutes. The most likely cause is the IAM role inline policy is "
                    f"missing cloudwatch:PutMetricData; CloudWatch silently drops the "
                    f"metric on IAM deny. Re-check the inline policy and re-run the job."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint runs two checks: most-recent Glue run state is SUCCEEDED, and `cloudwatch.get_metric_statistics` returns at least one datapoint for `labrun/OutputRowCount` with `JobName=JOB_NAME` in the last 30 minutes. If the run succeeded but no datapoint exists, the script's `boto3.client('cloudwatch').put_metric_data` call was denied silently; CloudWatch swallows IAM denies on the metric path. Open the role's inline policy and confirm there is a statement with `Action: cloudwatch:PutMetricData` and `Resource: "*"`.

</details>

<details><summary>Hint 2 (direction)</summary>

Six calls for the student in this cell: `s3.create_bucket`, two `s3.put_object` calls for the good and broken CSVs, one more `s3.put_object` for the script, `iam.create_role`, `iam.put_role_policy`, `glue.create_job`, and `glue.start_job_run`. The trust policy, inline policy, and script body are already built as constants in the cell; you just pass them to the boto3 calls. The 15-second IAM propagation sleep is already wired between role attach and job create. The 60-second metric-propagation sleep is already wired between job success and checkpoint.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1 (boto3 calls only; the script body, trust policy, and inline policy are already defined as constants in the cell):

```python
s3.create_bucket(Bucket=BUCKET_NAME)

s3.put_object(Bucket=BUCKET_NAME, Key="good/orders.csv", Body=good_csv)
s3.put_object(Bucket=BUCKET_NAME, Key="broken/orders.csv", Body=broken_csv)

s3.put_object(
    Bucket=BUCKET_NAME,
    Key=SCRIPT_KEY,
    Body=PYSPARK_SCRIPT.encode("utf-8"),
)

iam.create_role(
    RoleName=ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(inline_policy),
)

glue.create_job(
    Name=JOB_NAME,
    Role=role_arn,
    Command={
        "Name": "glueetl",
        "ScriptLocation": script_uri,
        "PythonVersion": "3",
    },
    DefaultArguments={
        "--BUCKET_NAME": BUCKET_NAME,
        "--INPUT_PREFIX": "good",
        "--enable-metrics": "true",
    },
    GlueVersion="4.0",
    WorkerType="G.1X",
    NumberOfWorkers=2,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

run = glue.start_job_run(JobName=JOB_NAME)
run_id = run["JobRunId"]
```

If `glue.create_job` returns `InvalidInputException: User: ... is not authorized to perform: iam:PassRole`, your labrun-test IAM user needs `iam:PassRole`; `IAMFullAccess` already grants it. If Checkpoint 1 fails on the missing datapoint after a SUCCEEDED run, the inline policy is missing the `cloudwatch:PutMetricData` statement; re-attach it with `iam.put_role_policy` and re-run the job (the metric is published only at run time, so a redeploy of the role mid-run does not help).

</details>


**Wallet check.** This cell is the first one that materially moves the bill. The good-data Glue run at 2 G.1X workers bills at $0.44 per DPU-hour with a 10-minute minimum, so about $0.15. CloudWatch custom metrics are free under the 10-metric allowance; you are publishing exactly one so you are well within it. S3 storage for ~3 MB of CSV is rounding error. Total so far: about 15 cents.

## Task 2: Create the CloudWatch alarm on labrun/OutputRowCount

One API call lands the alarm. The shape is:

```python
cloudwatch.put_metric_alarm(
    AlarmName=ALARM_NAME,
    Namespace=METRIC_NAMESPACE,
    MetricName=METRIC_NAME,
    Dimensions=[{"Name": "JobName", "Value": JOB_NAME}],
    Statistic="Maximum",
    ComparisonOperator="LessThanThreshold",
    Threshold=ALARM_THRESHOLD,
    EvaluationPeriods=ALARM_EVAL_PERIODS,
    Period=ALARM_PERIOD_SECONDS,
    TreatMissingData="notBreaching",
    AlarmActions=[TOPIC_ARN],  # set after we create the topic in Task 3
    ...
)
```

Two subtle choices worth understanding before the cert:

1. **`LessThanThreshold` not `GreaterThanThreshold`.** The alarm fires when the metric drops below 1000 rows. A silent failure that writes 47 rows trips it; a healthy 50,000-row run keeps the alarm in `OK`. Most CloudWatch examples threshold from below (high CPU, high latency); this lab is one of the legitimate "threshold from above" cases.
2. **`TreatMissingData="notBreaching"`.** With a 60-second period and a job that runs once a day, the metric will be missing for most of the day. `notBreaching` (or `ignore`) keeps the alarm in `OK` between runs. `breaching` would page on-call every minute the job is not running. The cert likes this question.

The Task 3 cell creates the SNS topic and adds it as `AlarmActions` via `put_metric_alarm` (which is idempotent: calling it again with new fields rewrites the alarm). For this task you create the alarm first with no actions, then in Task 3 re-call `put_metric_alarm` with the topic ARN.

Checkpoint 2 asserts the alarm exists with the right `MetricName`, `Namespace`, `Threshold`, `ComparisonOperator`, `EvaluationPeriods`, `Period`, and a `StateValue` in `{INSUFFICIENT_DATA, OK}` (a freshly created alarm is `INSUFFICIENT_DATA` until at least one datapoint arrives, and the good-data run from Task 1 likely already pushed it to `OK`).


In [ ]:
# NBVAL_SKIP
# Task 2: Create the CloudWatch alarm. No AlarmActions yet; Task 3 wires SNS.

cloudwatch = boto3.client(
    "cloudwatch",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the alarm with cloudwatch.put_metric_alarm(
#   AlarmName=ALARM_NAME,
#   AlarmDescription="Fires when Glue output row count drops below 1000",
#   Namespace=METRIC_NAMESPACE,
#   MetricName=METRIC_NAME,
#   Dimensions=[{"Name": "JobName", "Value": JOB_NAME}],
#   Statistic="Maximum",
#   ComparisonOperator="LessThanThreshold",
#   Threshold=ALARM_THRESHOLD,
#   EvaluationPeriods=ALARM_EVAL_PERIODS,
#   Period=ALARM_PERIOD_SECONDS,
#   TreatMissingData="notBreaching",
#   ActionsEnabled=True,
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).

# Resolve the alarm ARN now that the alarm exists; we need it for the
# EventBridge rule event pattern in Task 3.
desc = cloudwatch.describe_alarms(AlarmNames=[ALARM_NAME])
alarms = desc.get("MetricAlarms", [])
if not alarms:
    raise RuntimeError(
        f"Alarm {ALARM_NAME} was not found after put_metric_alarm. "
        f"Inspect the CloudWatch console and re-run this cell."
    )
ALARM_ARN = alarms[0]["AlarmArn"]
print(f"Alarm created: {ALARM_NAME}")
print(f"Alarm ARN: {ALARM_ARN}")
print(f"  Threshold: {ALARM_THRESHOLD} rows")
print(f"  Period: {ALARM_PERIOD_SECONDS}s, EvaluationPeriods: {ALARM_EVAL_PERIODS}")
print(f"  ComparisonOperator: LessThanThreshold")
print(f"  Initial StateValue: {alarms[0].get('StateValue')}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: alarm exists with the required shape and StateValue is one
# of INSUFFICIENT_DATA or OK (a fresh alarm starts INSUFFICIENT_DATA and
# transitions to OK once the good-data datapoint lands).

REQUIRED_ALARM_STATES = {"INSUFFICIENT_DATA", "OK"}


def checkpoint_2(session):
    try:
        cw_client = boto3.client(
            "cloudwatch",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        resp = cw_client.describe_alarms(AlarmNames=[ALARM_NAME])
        alarms = resp.get("MetricAlarms", [])
        if not alarms:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Alarm {ALARM_NAME} does not exist. Run the Task 2 cell."
                ),
            )
        alarm = alarms[0]

        if alarm.get("MetricName") != METRIC_NAME:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Alarm MetricName is {alarm.get('MetricName')!r}, "
                    f"expected {METRIC_NAME!r}."
                ),
            )
        if alarm.get("Namespace") != METRIC_NAMESPACE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Alarm Namespace is {alarm.get('Namespace')!r}, "
                    f"expected {METRIC_NAMESPACE!r}."
                ),
            )
        if alarm.get("Threshold") != ALARM_THRESHOLD:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Alarm Threshold is {alarm.get('Threshold')!r}, "
                    f"expected {ALARM_THRESHOLD!r}."
                ),
            )
        if alarm.get("ComparisonOperator") != "LessThanThreshold":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Alarm ComparisonOperator is {alarm.get('ComparisonOperator')!r}, "
                    f"expected 'LessThanThreshold'."
                ),
            )
        if alarm.get("EvaluationPeriods") != ALARM_EVAL_PERIODS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Alarm EvaluationPeriods is {alarm.get('EvaluationPeriods')!r}, "
                    f"expected {ALARM_EVAL_PERIODS!r}."
                ),
            )
        if alarm.get("Period") != ALARM_PERIOD_SECONDS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Alarm Period is {alarm.get('Period')!r}, "
                    f"expected {ALARM_PERIOD_SECONDS!r}."
                ),
            )
        state_value = alarm.get("StateValue", "UNKNOWN")
        if state_value not in REQUIRED_ALARM_STATES:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Alarm StateValue is {state_value!r}, expected one of "
                    f"{sorted(REQUIRED_ALARM_STATES)}. A fresh alarm starts "
                    f"INSUFFICIENT_DATA and transitions to OK once the good-data "
                    f"datapoint lands. If StateValue is ALARM, the broken-data "
                    f"run from Task 4 likely already fired; reset the alarm and "
                    f"re-run the good-data job."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks the alarm definition in order: MetricName, Namespace, Threshold, ComparisonOperator, EvaluationPeriods, Period, then StateValue. Read the failure reason. The most common miss on the first run is `ComparisonOperator` left at the API default (`GreaterThanThreshold`); for this lab it must be `LessThanThreshold` so the alarm fires when the row count drops below 1000. The next most common miss is `Threshold` passed as a string `"1000"` instead of the number `1000`; boto3 accepts both at create time but `describe_alarms` returns the number.

</details>

<details><summary>Hint 2 (direction)</summary>

One API call: `cloudwatch.put_metric_alarm`. All eleven parameters are listed inline in the cell. Pass `Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]` so the orphan scan can see it. Do not pass `AlarmActions` yet; Task 3 wires the SNS topic in after we create the topic. `ActionsEnabled=True` is the default but pass it explicitly to make the cert-style intent clear.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
cloudwatch.put_metric_alarm(
    AlarmName=ALARM_NAME,
    AlarmDescription="Fires when Glue output row count drops below 1000",
    Namespace=METRIC_NAMESPACE,
    MetricName=METRIC_NAME,
    Dimensions=[{"Name": "JobName", "Value": JOB_NAME}],
    Statistic="Maximum",
    ComparisonOperator="LessThanThreshold",
    Threshold=ALARM_THRESHOLD,
    EvaluationPeriods=ALARM_EVAL_PERIODS,
    Period=ALARM_PERIOD_SECONDS,
    TreatMissingData="notBreaching",
    ActionsEnabled=True,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

If `put_metric_alarm` returns `ValidationError: The parameter MetricName must be specified`, you passed `Metrics` (the metric-math array) instead of the flat `MetricName`/`Namespace`/`Dimensions` triple; pick one shape, not both. If the checkpoint reports `StateValue=ALARM` on the first run, the good-data job from Task 1 did not actually publish a datapoint (re-check the role's `cloudwatch:PutMetricData` grant) and the alarm flipped from `INSUFFICIENT_DATA` straight to `ALARM` on `TreatMissingData=breaching`; re-run Task 1.

</details>


**Wallet check.** Still about 15 cents. CloudWatch alarms are free under the 10-alarm allowance; this is your first one. You would have to leave eleven alarms running to start paying for them ($0.10 each per month after the free tier). Custom metrics: free under 10.

## Task 3: Create the SNS topic, wire AlarmActions, create the EventBridge rule, and target SNS

Four pieces here. The order matters because EventBridge needs the alarm ARN and the SNS topic ARN to exist before the rule can target either.

1. **SNS topic** named `labrun-cloudwatch-pipeline-monitoring-topic`. `sns.create_topic` is idempotent on the name; it returns the topic ARN whether the topic is new or already existed. Tag it with the lab tag.
2. **SNS HTTPS subscription** to `DEFAULT_SUBSCRIPTION_ENDPOINT`. HTTPS subscriptions require confirmation: SNS posts a `SubscriptionConfirmation` JSON to the endpoint, and you have to visit a URL inside that JSON to confirm. For this lab the subscription will stay in `PendingConfirmation` (the test endpoint is a Beeceptor-style sink that displays the JSON but does not auto-confirm). That is fine: Checkpoint 4 inspects `NumberOfMessagesPublished`, not `NumberOfMessagesDelivered`. If you want to receive the actual page in your inbox, swap `DEFAULT_SUBSCRIPTION_ENDPOINT` for an email address and use `Protocol="email"`; you will still get one confirmation email before any real pages flow.
3. **Wire SNS as an `AlarmActions` target** by calling `put_metric_alarm` a second time. This is defense in depth: even if EventBridge breaks, the alarm itself pages SNS directly. The cert calls this pattern out as a recommended practice.
4. **EventBridge rule** named `labrun-cloudwatch-pipeline-monitoring-eb-rule` on the default bus. The event pattern matches `source: ["aws.cloudwatch"]`, `detail-type: ["CloudWatch Alarm State Change"]`, and `resources: [<alarm ARN>]`. Add one target: the SNS topic ARN. EventBridge needs `sns:Publish` permission to deliver to the topic; in `us-east-1`, the default topic policy grants `events.amazonaws.com` permission to publish via the `AWSEvents_*` SID without an explicit policy edit, but you can attach an explicit `SetTopicAttributes` policy if your account has hardened the default.

Checkpoint 3 walks the EventBridge rule's `EventPattern`, asserts the rule is `ENABLED`, and confirms `list_targets_by_rule` returns exactly one target with `Arn == sns_topic_arn`.


In [ ]:
# NBVAL_SKIP
# Task 3: SNS topic + subscription, wire AlarmActions, create the EventBridge
# rule and target SNS.

sns = boto3.client(
    "sns",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
events = boto3.client(
    "events",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the SNS topic with topic_resp = sns.create_topic(
#   Name=TOPIC_NAME,
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ). Capture TOPIC_ARN = topic_resp["TopicArn"].

print(f"SNS topic created: {TOPIC_NAME}")
print(f"  Topic ARN: {TOPIC_ARN}")

# Update the manifest entry now that we know the real ARN. The deletion CLI
# string is rewritten so the cleanup printout is accurate.
for r in CLEANUP_MANIFEST:
    if r.type == "sns_topic":
        r.cli_delete_command = f"aws sns delete-topic --topic-arn {TOPIC_ARN}"
        break

# YOUR CODE: Create the HTTPS subscription with sub_resp = sns.subscribe(
#   TopicArn=TOPIC_ARN,
#   Protocol="https",
#   Endpoint=DEFAULT_SUBSCRIPTION_ENDPOINT,
#   ReturnSubscriptionArn=True,
# ). Capture SUBSCRIPTION_ARN = sub_resp["SubscriptionArn"].

print(f"SNS subscription created: {SUBSCRIPTION_ARN}")
print("HTTPS subs require confirmation: SNS posted a SubscriptionConfirmation")
print("JSON to the endpoint. The subscription will stay in PendingConfirmation")
print("for this lab; that does not block Checkpoint 4 which inspects published")
print("messages, not delivered messages.")

# Update the manifest entry for the subscription. The id is rewritten so the
# inline _LabAdapter can call sns.unsubscribe(SubscriptionArn=...) at cleanup.
for r in CLEANUP_MANIFEST:
    if r.type == "sns_subscription":
        r.id = SUBSCRIPTION_ARN
        r.cli_delete_command = f"aws sns unsubscribe --subscription-arn {SUBSCRIPTION_ARN}"
        break

# Re-call put_metric_alarm with the SNS topic in AlarmActions. put_metric_alarm
# is idempotent: calling it again with the same AlarmName rewrites the alarm
# definition rather than creating a duplicate.
cloudwatch = boto3.client(
    "cloudwatch",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
cloudwatch.put_metric_alarm(
    AlarmName=ALARM_NAME,
    AlarmDescription="Fires when Glue output row count drops below 1000",
    Namespace=METRIC_NAMESPACE,
    MetricName=METRIC_NAME,
    Dimensions=[{"Name": "JobName", "Value": JOB_NAME}],
    Statistic="Maximum",
    ComparisonOperator="LessThanThreshold",
    Threshold=ALARM_THRESHOLD,
    EvaluationPeriods=ALARM_EVAL_PERIODS,
    Period=ALARM_PERIOD_SECONDS,
    TreatMissingData="notBreaching",
    ActionsEnabled=True,
    AlarmActions=[TOPIC_ARN],
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
print(f"AlarmActions on {ALARM_NAME} now include the SNS topic.")

event_pattern = {
    "source": ["aws.cloudwatch"],
    "detail-type": ["CloudWatch Alarm State Change"],
    "resources": [ALARM_ARN],
}

# YOUR CODE: Create the EventBridge rule with events.put_rule(
#   Name=RULE_NAME,
#   EventPattern=json.dumps(event_pattern),
#   State="ENABLED",
#   Description="Routes CloudWatch alarm state changes to SNS",
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).

# YOUR CODE: Wire the SNS topic as the rule's only target:
#   events.put_targets(
#       Rule=RULE_NAME,
#       Targets=[{"Id": "1", "Arn": TOPIC_ARN}],
#   ).

print(f"EventBridge rule created: {RULE_NAME}")
print(f"  Target 1: {TOPIC_ARN}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: EventBridge rule exists with the correct EventPattern keys,
# State==ENABLED, and exactly one target with Arn == TOPIC_ARN.


def checkpoint_3(session):
    try:
        events_client = boto3.client(
            "events",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            rule = events_client.describe_rule(Name=RULE_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"EventBridge rule {RULE_NAME} does not exist. Run the Task 3 cell.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if rule.get("State") != "ENABLED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Rule State is {rule.get('State')!r}, expected 'ENABLED'. "
                    f"Call events.enable_rule(Name=RULE_NAME) or re-run put_rule "
                    f"with State='ENABLED'."
                ),
            )

        pattern_raw = rule.get("EventPattern")
        if not pattern_raw:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Rule {RULE_NAME} has no EventPattern. Re-call events.put_rule "
                    f"with EventPattern=json.dumps({{...}})."
                ),
            )
        try:
            pattern = json.loads(pattern_raw)
        except json.JSONDecodeError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"Rule EventPattern is not valid JSON: {e}",
            )

        if pattern.get("source") != ["aws.cloudwatch"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"EventPattern.source is {pattern.get('source')!r}, "
                    f"expected ['aws.cloudwatch']."
                ),
            )
        if pattern.get("detail-type") != ["CloudWatch Alarm State Change"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"EventPattern['detail-type'] is {pattern.get('detail-type')!r}, "
                    f"expected ['CloudWatch Alarm State Change']."
                ),
            )
        resources_field = pattern.get("resources", [])
        if ALARM_ARN not in resources_field:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"EventPattern.resources is {resources_field!r}, expected "
                    f"to contain {ALARM_ARN!r}. Re-call events.put_rule with "
                    f"the alarm ARN in the resources list."
                ),
            )

        targets_resp = events_client.list_targets_by_rule(Rule=RULE_NAME)
        targets = targets_resp.get("Targets", [])
        if len(targets) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Rule {RULE_NAME} has {len(targets)} target(s), expected exactly 1."
                ),
            )
        target_arn = targets[0].get("Arn")
        if target_arn != TOPIC_ARN:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Rule target Arn is {target_arn!r}, expected {TOPIC_ARN!r}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks the rule definition: State must be `ENABLED`, EventPattern.source must be `["aws.cloudwatch"]`, EventPattern['detail-type'] must be `["CloudWatch Alarm State Change"]`, EventPattern.resources must include the alarm ARN, and `list_targets_by_rule` must return exactly one target with `Arn == TOPIC_ARN`. The most common miss is forgetting `json.dumps` on the event pattern; `events.put_rule` requires a JSON string, not a Python dict. The second most common miss is the `detail-type` key in Python; do not use `detail_type` (snake_case is invalid here), the literal string is `detail-type`.

</details>

<details><summary>Hint 2 (direction)</summary>

Four API calls in this cell. `sns.create_topic` returns the topic ARN in the `TopicArn` key. `sns.subscribe` with `ReturnSubscriptionArn=True` returns the real subscription ARN even while it sits in `PendingConfirmation`; without that flag, AWS returns the literal string `"pending confirmation"` for HTTPS/email subscriptions and your cleanup adapter cannot unsubscribe it. `events.put_rule` with the JSON-stringified `event_pattern` and `State="ENABLED"` creates the rule. `events.put_targets` adds the SNS topic with `Id="1"`. The `put_metric_alarm` re-call that wires `AlarmActions` is already in the cell.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3:

```python
topic_resp = sns.create_topic(
    Name=TOPIC_NAME,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
TOPIC_ARN = topic_resp["TopicArn"]

sub_resp = sns.subscribe(
    TopicArn=TOPIC_ARN,
    Protocol="https",
    Endpoint=DEFAULT_SUBSCRIPTION_ENDPOINT,
    ReturnSubscriptionArn=True,
)
SUBSCRIPTION_ARN = sub_resp["SubscriptionArn"]

events.put_rule(
    Name=RULE_NAME,
    EventPattern=json.dumps(event_pattern),
    State="ENABLED",
    Description="Routes CloudWatch alarm state changes to SNS",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
events.put_targets(
    Rule=RULE_NAME,
    Targets=[{"Id": "1", "Arn": TOPIC_ARN}],
)
```

If `events.put_targets` returns `FailedEntries` with `Error: "events.amazonaws.com cannot publish"`, the SNS topic policy is missing the default `AWSEvents_*` SID. Fix it with `sns.set_topic_attributes` and a policy that allows `events.amazonaws.com` to `sns:Publish` on the topic ARN. If `sns.subscribe` returns `SubscriptionArn=="PendingConfirmation"`, you forgot `ReturnSubscriptionArn=True`; the cleanup adapter swallows `InvalidParameter` on unsubscribe for that pending-string case, but you lose the ability to script the unsubscribe.

</details>


**Wallet check.** Still about 15 cents. SNS topics are free; the first 1,000 HTTPS publishes per month are free. EventBridge rules on the default bus matching AWS service events (like CloudWatch alarm state changes) are free. The metric custom-name count is still 1. No new line items appeared on the bill in this task.

## Task 4: Trigger the alarm with a broken run and confirm SNS published a message

The proof step. The Glue job's `DefaultArguments` carries `--INPUT_PREFIX=good`; you override it for this run only with `Arguments={"--INPUT_PREFIX": "broken"}` on `start_job_run`. The broken dataset is 50 rows, well below the 1000 threshold, so:

1. The run finishes SUCCEEDED (it is a valid Glue run; the data is just sparse).
2. The script publishes `OutputRowCount=50` for `JobName=JOB_NAME`.
3. CloudWatch evaluates the alarm on the next 60-second period boundary. Because `Statistic="Maximum"` over the period and the only datapoint is 50, the alarm transitions to `ALARM`.
4. The state change fires both the `AlarmActions` direct path (SNS receives a notification from the alarm) and the EventBridge rule (SNS receives a second notification from the rule). Defense in depth, as Task 3 noted.
5. `cloudwatch.get_metric_statistics` on the AWS-namespace metric `AWS/SNS NumberOfMessagesPublished` (dimension `TopicName=TOPIC_NAME`) shows at least one message in the last 5 minutes.

The poll loop in the cell waits up to 5 minutes for the alarm to transition to `ALARM`. In practice it takes 60 to 120 seconds: the metric arrives at the end of the job, then CloudWatch evaluates on the next period boundary, then state-change propagates.

Checkpoint 4 asserts the alarm reached `ALARM` and `NumberOfMessagesPublished` on the SNS topic shows at least one message published in the last 5 minutes.


In [ ]:
# NBVAL_SKIP
# Task 4: Trigger the alarm by running the Glue job against the broken
# dataset, then poll the alarm and SNS metrics until both confirm a page.

glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
cloudwatch = boto3.client(
    "cloudwatch",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Start the broken-data run by overriding the default INPUT_PREFIX:
#   broken_run = glue.start_job_run(
#       JobName=JOB_NAME,
#       Arguments={"--INPUT_PREFIX": "broken"},
#   )
#   broken_run_id = broken_run["JobRunId"]

print(f"Broken-data run submitted; run id will be printed below.")
print("Asking Glue to spin up workers; this takes a minute or so...")

TERMINAL_STATES = {"SUCCEEDED", "FAILED", "STOPPED", "TIMEOUT"}
deadline = time.time() + 720
last_state = None
run_resp = None
while time.time() < deadline:
    run_resp = glue.get_job_run(JobName=JOB_NAME, RunId=broken_run_id)
    state = run_resp["JobRun"]["JobRunState"]
    if state != last_state:
        print(f"  state: {state}")
        last_state = state
    if state in TERMINAL_STATES:
        break
    time.sleep(15)
else:
    raise RuntimeError(
        f"Broken-data run {broken_run_id} did not reach a terminal state within "
        f"12 minutes. Inspect the run in the AWS Glue console."
    )

final_state = run_resp["JobRun"]["JobRunState"]
if final_state != "SUCCEEDED":
    err = run_resp["JobRun"].get("ErrorMessage", "(no error message)")
    raise RuntimeError(
        f"Broken-data run finished {final_state}: {err}. The 50-row dataset "
        f"should still produce a SUCCEEDED run; inspect the Glue console."
    )
print(f"Broken-data run finished SUCCEEDED.")
print("Polling the alarm; CloudWatch needs 60 to 120 seconds to evaluate...")

alarm_deadline = time.time() + 300  # 5 minutes
final_alarm_state = None
while time.time() < alarm_deadline:
    desc = cloudwatch.describe_alarms(AlarmNames=[ALARM_NAME])
    state_value = desc["MetricAlarms"][0].get("StateValue", "UNKNOWN")
    if state_value != final_alarm_state:
        print(f"  alarm StateValue: {state_value}")
        final_alarm_state = state_value
    if state_value == "ALARM":
        break
    time.sleep(15)

if final_alarm_state != "ALARM":
    print(f"Alarm did not reach ALARM within 5 minutes; current state {final_alarm_state}.")
    print("Checkpoint 4 will report the failure with details.")
else:
    print("Alarm transitioned to ALARM. Sleeping 30s so SNS metrics catch up...")
    time.sleep(30)


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: alarm state is ALARM and SNS NumberOfMessagesPublished for
# the topic shows at least one message in the last 5 minutes.


def checkpoint_4(session):
    try:
        cw_client = boto3.client(
            "cloudwatch",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        desc = cw_client.describe_alarms(AlarmNames=[ALARM_NAME])
        alarms = desc.get("MetricAlarms", [])
        if not alarms:
            return CheckpointResult(
                status="fail",
                error_reason=f"Alarm {ALARM_NAME} does not exist. Run earlier tasks first.",
            )
        state_value = alarms[0].get("StateValue", "UNKNOWN")
        if state_value != "ALARM":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Alarm StateValue is {state_value!r}, expected 'ALARM'. The "
                    f"broken-data Glue run should have published OutputRowCount=50 "
                    f"which is below the 1000 threshold. Wait up to 2 more minutes "
                    f"for CloudWatch to evaluate, then re-run this checkpoint."
                ),
            )

        end_time = datetime.now(timezone.utc)
        start_time = end_time - timedelta(minutes=5)
        sns_stats = cw_client.get_metric_statistics(
            Namespace="AWS/SNS",
            MetricName="NumberOfMessagesPublished",
            Dimensions=[{"Name": "TopicName", "Value": TOPIC_NAME}],
            StartTime=start_time,
            EndTime=end_time,
            Period=60,
            Statistics=["Sum"],
        )
        datapoints = sns_stats.get("Datapoints", [])
        total_published = sum(dp.get("Sum", 0) for dp in datapoints)
        if total_published < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AWS/SNS NumberOfMessagesPublished for topic {TOPIC_NAME} shows "
                    f"{total_published} message(s) in the last 5 minutes, expected "
                    f">= 1. The alarm is in ALARM state but SNS did not receive a "
                    f"publish. Confirm AlarmActions on the alarm includes the topic "
                    f"ARN and the EventBridge rule target is the topic ARN."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint runs two checks: alarm `StateValue == "ALARM"` and `AWS/SNS NumberOfMessagesPublished` for the topic shows at least one message in the last 5 minutes. If the alarm is still `INSUFFICIENT_DATA` or `OK`, the broken-data datapoint either never arrived (re-check Checkpoint 1's diagnosis: the role needs `cloudwatch:PutMetricData`) or arrived after the polling deadline (wait two more minutes and re-run the checkpoint). If the alarm is in `ALARM` but SNS shows zero publishes, the alarm's `AlarmActions` is missing the topic ARN; re-run the `put_metric_alarm` re-call from Task 3.

</details>

<details><summary>Hint 2 (direction)</summary>

Two lines for the student in this task: `broken_run = glue.start_job_run(JobName=JOB_NAME, Arguments={"--INPUT_PREFIX": "broken"})` and `broken_run_id = broken_run["JobRunId"]`. The poll loop on the run and the poll loop on the alarm state are already wired. The 30-second post-ALARM sleep is there so the SNS-side `NumberOfMessagesPublished` metric (which is itself a CloudWatch metric and has its own propagation delay) has time to register.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4 (the poll loops below the call are unchanged):

```python
broken_run = glue.start_job_run(
    JobName=JOB_NAME,
    Arguments={"--INPUT_PREFIX": "broken"},
)
broken_run_id = broken_run["JobRunId"]
```

If the alarm fires but the SNS metric still reads 0, AWS CloudWatch metric propagation for `AWS/SNS NumberOfMessagesPublished` can take up to 5 minutes; sleep another 60 seconds and re-run the checkpoint. If the alarm never transitions and you see `StateValue=OK` after the broken run, your `Statistic` on `put_metric_alarm` is probably `Average` instead of `Maximum`; the broken datapoint averaged with the older good datapoint can stay above 1000 in the same evaluation window. `Maximum` (or `Minimum`) ensures one bad datapoint trips the alarm.

</details>


**Wallet check.** Another Glue ETL run at 2 G.1X workers for the 10-minute Glue minimum is another $0.15. Running total: about 30 cents. CloudWatch alarm state changes and SNS publishes at this volume are still free. The next cell deletes everything so the bill stops here.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. Lab 10 has no critical (hourly-billed) resources.
#
# labrun-checks 0.3.0 ships AWS adapters for s3_bucket, iam_role,
# glue_database, glue_table, kinesis_stream. It does NOT yet ship adapters
# for glue_job, cloudwatch_alarm, eventbridge_rule, sns_topic, or
# sns_subscription. An inline _LabAdapter subclass extending
# AwsCleanupAdapter adds them. When the package promotes these adapters in
# a future release, the wrapper can be removed and run_cleanup called
# against the default adapter.
#
# - cloudwatch_alarm: cloudwatch.delete_alarms(AlarmNames=[id]).
# - eventbridge_rule: events.remove_targets(Rule=id, Ids=["1"], Force=True)
#   then events.delete_rule(Name=id). Force=True so a rule with stuck
#   managed-target permissions still tears down.
# - sns_topic: sns.delete_topic(TopicArn=arn_resolved_from_id_or_TOPIC_ARN).
# - sns_subscription: sns.unsubscribe(SubscriptionArn=id). HTTPS subs that
#   stayed in PendingConfirmation return InvalidParameter on unsubscribe
#   because the ARN literal is "PendingConfirmation"; that error is
#   swallowed silently.
# - glue_job: glue.delete_job(JobName=id).

import sys
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    """AwsCleanupAdapter wrapper that adds Lab 10 inline resource types.

    Inlined types: cloudwatch_alarm, eventbridge_rule, sns_topic,
    sns_subscription, glue_job. Everything else (s3_bucket, iam_role)
    delegates to the bundled AwsCleanupAdapter unchanged.
    """

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def _client(self, service: str, credentials: dict):
        return boto3.client(
            service,
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "cloudwatch_alarm":
            client = self._client("cloudwatch", credentials)
            try:
                client.delete_alarms(AlarmNames=[resource.id])
            except ClientError as e:
                if e.response["Error"]["Code"] not in (
                    "ResourceNotFound", "ResourceNotFoundException"
                ):
                    raise
            return

        if resource.type == "eventbridge_rule":
            client = self._client("events", credentials)
            try:
                client.remove_targets(Rule=resource.id, Ids=["1"], Force=True)
            except ClientError as e:
                if e.response["Error"]["Code"] not in (
                    "ResourceNotFoundException", "InternalException"
                ):
                    raise
            try:
                client.delete_rule(Name=resource.id, Force=True)
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            return

        if resource.type == "sns_topic":
            client = self._client("sns", credentials)
            arn = TOPIC_ARN or f"arn:aws:sns:{REGION}:{ACCOUNT_ID}:{resource.id}"
            try:
                client.delete_topic(TopicArn=arn)
            except ClientError as e:
                if e.response["Error"]["Code"] not in ("NotFound", "NotFoundException"):
                    raise
            return

        if resource.type == "sns_subscription":
            client = self._client("sns", credentials)
            arn = resource.id
            if not arn or arn in ("pending", "PendingConfirmation"):
                # The subscription never received a real ARN; nothing to do.
                return
            try:
                client.unsubscribe(SubscriptionArn=arn)
            except ClientError as e:
                code_ = e.response["Error"]["Code"]
                # InvalidParameter fires when the ARN literal is the
                # placeholder "PendingConfirmation" string from the
                # ReturnSubscriptionArn=False default; swallow it.
                if code_ not in (
                    "NotFound", "NotFoundException", "InvalidParameter"
                ):
                    raise
            return

        if resource.type == "glue_job":
            client = self._client("glue", credentials)
            try:
                client.delete_job(JobName=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "EntityNotFoundException":
                    raise
            return

        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "cloudwatch_alarm":
            client = self._client("cloudwatch", credentials)
            resp = client.describe_alarms(AlarmNames=[resource.id])
            return bool(resp.get("MetricAlarms"))

        if resource.type == "eventbridge_rule":
            client = self._client("events", credentials)
            try:
                client.describe_rule(Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "sns_topic":
            client = self._client("sns", credentials)
            arn = TOPIC_ARN or f"arn:aws:sns:{REGION}:{ACCOUNT_ID}:{resource.id}"
            try:
                client.get_topic_attributes(TopicArn=arn)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] in ("NotFound", "NotFoundException"):
                    return False
                raise

        if resource.type == "sns_subscription":
            client = self._client("sns", credentials)
            arn = resource.id
            if not arn or arn in ("pending", "PendingConfirmation"):
                return False
            try:
                client.get_subscription_attributes(SubscriptionArn=arn)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] in (
                    "NotFound", "NotFoundException", "InvalidParameter"
                ):
                    return False
                raise

        if resource.type == "glue_job":
            client = self._client("glue", credentials)
            try:
                client.get_job(JobName=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise

        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before run_cleanup tears it down. S3 buckets cannot
# be deleted while they contain objects, and the Glue job wrote Parquet
# under out/good/ and out/broken/ plus the raw CSVs plus scripts/transform.py.
print("Emptying the S3 bucket before teardown...")
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

# Drop inline policies on the role before role-delete so the iam_role
# adapter does not block on attached policies.
iam_client = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    listed = iam_client.list_role_policies(RoleName=ROLE_NAME)
    for policy_name in listed.get("PolicyNames", []):
        iam_client.delete_role_policy(RoleName=ROLE_NAME, PolicyName=policy_name)
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchEntity":
        print(f"Inline policy detach for {ROLE_NAME} ran into: {e}. Continuing.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: $0.30 to $0.50.** Two Glue ETL job runs at 2 G.1X workers for the 10-minute Glue minimum are about $0.15 each, so $0.30 if both worked on the first try. A third run while you debug the role's `cloudwatch:PutMetricData` grant or the script's argument parsing pushes you toward $0.50. CloudWatch alarms, custom metrics, EventBridge rules, and SNS publishes at this volume are all free; you would have to leave eleven alarms running to start paying for them. CloudWatch does not support deleting custom metrics via API, so the `labrun/OutputRowCount` metric will age out after 14 months of no data; that costs nothing because you only published two datapoints and the always-free metric tier covers the storage. The cleanup cell above deleted the SNS subscription, the SNS topic, the EventBridge rule, the alarm, the Glue job, the IAM role, and the bucket so the bill stops accruing now.

## These are not graded. They are for you.

Three questions worth sitting with before the exam.

1. The alarm in this lab uses `Statistic="Maximum"` over a 60-second period. Walk through what changes if you set `Statistic="Average"`. Hint: in the cert-scenario world where the bad ETL pushes one 47-row datapoint into the same evaluation window as yesterday's healthy 50,000-row datapoint, the average is still way above 1000 and the alarm never fires. `Maximum` (or `Minimum` for "alert on any bad row") is the safer default for silent-failure alarms on counts.

2. This lab has two parallel paths from CloudWatch to SNS: the alarm's own `AlarmActions` and the EventBridge rule's target. Both fire on the same state change. Walk through two reasons the cert prefers the EventBridge path as the production pattern. Hint: EventBridge gives you filtering (route only `ALARM` transitions, not `OK`), fan-out to multiple targets (Lambda, Step Functions, SQS) without editing the alarm, and event-pattern composability across many alarms with one rule.

3. CloudWatch does not let you delete custom metrics via API; they age out after 14 months. Identify two ways this matters in production. Hint: cost (a runaway script that publishes a million distinct metric names is a budget event you cannot trim), and observability hygiene (the AWS console namespace fills with stale metrics from old jobs and dashboards drift). The fix is naming conventions and namespace discipline before the first `put_metric_data` call.

## Exam-Style Practice

**Q1.** A data engineer wants an alarm to fire when an overnight Glue job writes fewer than 1000 rows. The job runs once at 02:00 UTC and publishes one custom metric datapoint at the end. With `Period=60`, `EvaluationPeriods=1`, and `TreatMissingData="breaching"`, the alarm pages on-call all day even when the job has not run yet. What is the right fix?

A) Change `TreatMissingData` to `"notBreaching"` so missing periods between runs do not flip the alarm to ALARM.

B) Increase `EvaluationPeriods` to 1440 so the alarm only fires after 24 hours of bad data.

C) Move the alarm threshold to `GreaterThanThreshold` so missing data is treated as healthy.

D) Set `Statistic="SampleCount"` instead of `Sum` so missing periods do not count.

<details><summary>Show answer</summary>

**A**.

- A) Right. `TreatMissingData="notBreaching"` (or `"ignore"`) is the canonical pattern for batch-job metrics where data arrives once per day. `"breaching"` makes every missing 60-second window count as a bad datapoint, which is correct for "always-on" services but wrong for batch jobs.
- B) Wrong. With `EvaluationPeriods=1440` the alarm only fires after 24 hours of consecutive bad data, which means a bad overnight run does not page on-call until the next overnight run. The lag defeats the purpose.
- C) Wrong. Flipping the operator does not fix the missing-data behavior; the alarm would still evaluate on every period and treat missing data as bad.
- D) Wrong. `SampleCount` reports the number of datapoints, not the value; the alarm would always fire (sample count is 0 for missing periods, which is less than the threshold).

</details>

**Q2.** A data engineer wants to fan out CloudWatch alarm state changes to both an SNS topic for paging and a Lambda function for automatic remediation. The current setup uses the alarm's own `AlarmActions` field. Which option is the cleanest production pattern?

A) Add the Lambda function ARN to the alarm's `AlarmActions` list alongside the SNS topic ARN.

B) Create an EventBridge rule on `aws.cloudwatch` source matching `CloudWatch Alarm State Change` events for the alarm and add both the SNS topic and the Lambda as targets.

C) Subscribe the Lambda to the SNS topic so every page also triggers the Lambda.

D) Move both targets into `OKActions` instead of `AlarmActions` so the routing is symmetric.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Adding the Lambda to `AlarmActions` works but does not scale: every new target requires editing the alarm, and you cannot filter by transition type (alarms also fire `OKActions` and `InsufficientDataActions` separately). The cert prefers EventBridge for fan-out.
- B) Right. EventBridge rules give you filtering, fan-out, and composability across many alarms with one rule. The state-change event includes both `OldState` and `NewState`, so a rule can filter for `NewState=ALARM` and skip `NewState=OK`. Adding a third or fourth target is a `put_targets` call, not an alarm edit.
- C) Wrong. Subscribing Lambda to SNS works but coupling remediation to paging is fragile: a paused subscription, an SNS throttle, or a topic policy edit silently breaks remediation. EventBridge to multiple targets is independent.
- D) Wrong. `OKActions` fires on the OK transition, not on ALARM; this routes paging to the recovery direction. The fields are not symmetric.

</details>

**Q3.** A Glue ETL job publishes a custom CloudWatch metric `labrun/OutputRowCount` with dimension `JobName`. The script ran successfully but `cloudwatch.get_metric_statistics` returns no datapoints. The job's CloudWatch Logs show the `put_metric_data` call returned without raising. What is the most likely cause?

A) The custom metric namespace `labrun` is reserved; AWS rejects publishes silently to it.

B) The Glue job's IAM role is missing `cloudwatch:PutMetricData`; the metric path silently drops on IAM deny.

C) `get_metric_statistics` requires the `Statistic="SampleCount"` argument; the default is unsupported.

D) Custom metrics are visible only after a 14-month propagation window.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. There is no reserved namespace `labrun`; AWS reserves only the `AWS/*` prefix for first-party namespaces. Custom names work for any non-`AWS/` namespace.
- B) Right. `cloudwatch:PutMetricData` denies on the metric path return silently from the SDK's perspective in some runtime contexts (Glue's PySpark execution environment swallows the deny rather than raising), so the script appears to succeed but no datapoint lands. Adding the action to the role's inline policy fixes it.
- C) Wrong. `get_metric_statistics` requires `Statistics` (the parameter) but accepts any combination of `SampleCount`, `Sum`, `Average`, `Minimum`, `Maximum`; the default-statistic claim is false.
- D) Wrong. Custom metrics are visible within a minute of publication. The 14-month figure is the auto-expiration window, not propagation latency.

</details>
